# 🎙️ High-Performance Arabic Text-to-Speech Pipeline (Google Colab Edition)

**End-to-end robust training, dataset preprocessing, and ONNX export for Google Colab with full Google Drive persistence.**

### 🌟 Advanced Features
- ⚡ **Full GPU Harnessing**: Auto-detects Colab GPU tier (A100/V100/T4) and configures precision (`bf16-mixed` / `16-mixed`), cuDNN benchmarks, and PyTorch float32 matmul precision.
- 🛡️ **Robust Audio & Text Sanitization**: Detects corrupted audio, zero-byte files, silent clips, and invalid format headers. Skips bad files alongside their transcriptions automatically.
- 🗃️ **Universal Dataset Handler**: Supports HuggingFace datasets, direct archives (`.zip`, `.tar.gz`), and raw audio (`.mp3`, `.flac`, `.wav`, `.ogg`, `.m4a`). Converts all audio to high-fidelity mono PCM WAV.
- 📦 **ONNX Streaming Export**: Cell dedicated to exporting `encoder`, `decoder_stream`, and `vocoder_stream` models with optional INT8 dynamic quantization directly to Google Drive.
- 🔄 **Resilient Resume**: Phase markers & automatic checkpoint selection ensure instant training resumption after Colab disconnects without repeating data processing.

---

## ⚙️ Cell 1 — Global Configuration

Configure project paths and training hyperparameters here.

In [ ]:
#@title ⚙️ Configuration {display-mode: "form"}

#@markdown ### 📂 Google Drive Paths
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/tts_project"  #@param {type:"string"}

#@markdown ### 🗃️ HuggingFace Dataset
HF_DATASET_ID = "Mohamad-I8/AseelArabicDataset"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

#@markdown ### 🔧 GitHub Repository
GITHUB_REPO_URL = "https://github.com/shamsorachdi62/repo.git"  #@param {type:"string"}

#@markdown ### 🏋️ Training Parameters
EXPERIMENT_NAME = "saeed"  #@param {type:"string"}
BATCH_SIZE = 16  #@param {type:"integer"}
MAX_STEPS = 300000  #@param {type:"integer"}
SAMPLE_RATE = 24000  #@param {type:"integer"}
PREPROCESS_WORKERS = 2  #@param {type:"integer"}

# ────────────────────────────────────────────────────
# Derived paths & Phase Markers
# ────────────────────────────────────────────────────
import os

DRIVE_CHECKPOINTS   = os.path.join(DRIVE_PROJECT_ROOT, "checkpoints")
DRIVE_LOGS          = os.path.join(DRIVE_PROJECT_ROOT, "logs")
DRIVE_PREPROCESSED  = os.path.join(DRIVE_PROJECT_ROOT, "preprocessed_data")
DRIVE_RAW_DATASET   = os.path.join(DRIVE_PROJECT_ROOT, "raw_dataset")
DRIVE_CONVERTED_WAV = os.path.join(DRIVE_PROJECT_ROOT, "raw_dataset", "wav")
DRIVE_DATA_STATS    = os.path.join(DRIVE_PROJECT_ROOT, "data_stats")
DRIVE_ONNX_EXPORT   = os.path.join(DRIVE_PROJECT_ROOT, "onnx_exports")

LOCAL_REPO          = "/content/repo"
LOCAL_DATA_DIR      = os.path.join(LOCAL_REPO, "data", EXPERIMENT_NAME)

MARKER_DIR             = os.path.join(DRIVE_PROJECT_ROOT, ".markers")
MARKER_DOWNLOAD_DONE   = os.path.join(MARKER_DIR, "01_download_done")
MARKER_CONVERT_DONE    = os.path.join(MARKER_DIR, "02_convert_done")
MARKER_METADATA_DONE   = os.path.join(MARKER_DIR, "03_metadata_done")
MARKER_PREPROCESS_DONE = os.path.join(MARKER_DIR, "04_preprocess_done")
MARKER_STATS_DONE      = os.path.join(MARKER_DIR, "05_stats_done")

print("✅ Configuration loaded successfully.")

## 📌 Cell 2 — Mount Google Drive & Environment Setup

In [ ]:
#@title 📌 Mount Google Drive {display-mode: "form"}

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

for d in [
    DRIVE_PROJECT_ROOT, DRIVE_CHECKPOINTS, DRIVE_LOGS,
    DRIVE_PREPROCESSED, DRIVE_RAW_DATASET, DRIVE_CONVERTED_WAV,
    DRIVE_DATA_STATS, DRIVE_ONNX_EXPORT, MARKER_DIR
]:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive mounted and directories initialized.")
print(f"   Project root: {DRIVE_PROJECT_ROOT}")

## ⚡ Cell 3 — GPU Hardware Acceleration & Dependencies Setup

Detects GPU capability (A100 vs T4/V100), tunes PyTorch CUDA backends for max performance, and safely installs model requirements without corrupting Colab's CUDA runtime.

In [ ]:
#@title ⚡ GPU Acceleration & Dependencies {display-mode: "form"}

import subprocess, sys, os, torch

# ── GPU Hardware Detection & Speed Optimizations ──
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"⚡ GPU Detected: {gpu_name} (Compute Capability {cap[0]}.{cap[1]})")
    
    # Maximize GPU utilization
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    
    if cap[0] >= 8:  # Ampere architecture (A100, L4)
        OPTIMAL_PRECISION = "bf16-mixed"
        print("   🚀 Ampere GPU detected! Enabled bfloat16 mixed precision for maximum throughput.")
    else:  # Turing / Volta architecture (T4, V100)
        OPTIMAL_PRECISION = "16-mixed"
        print("   ⚡ Tensor Core GPU detected! Enabled fp16 mixed precision.")
else:
    OPTIMAL_PRECISION = "32"
    print("⚠️  No GPU detected! Running on CPU mode (slow).")

# ── Clone Repository ──
if os.path.isdir(LOCAL_REPO):
    print("\n📂 Repository exists. Pulling latest code...")
    subprocess.run(["git", "pull"], cwd=LOCAL_REPO, check=False)
else:
    print("\n📥 Cloning repository...")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, LOCAL_REPO], check=True)
    print("✅ Repository cloned.")

# ── Dependency Installation ──
SAFE_DEPS = [
    "lightning>=2.0.0",
    "torchmetrics>=0.11.4",
    "nnaudio>=0.3.3",
    "pyworld>=0.3.4",
    "hydra-core>=1.3.2",
    "hydra-colorlog>=1.2.0",
    "rootutils>=1.0.7",
    "rich>=13.7.1",
    "librosa==0.9.2",
    "einops>=0.8.0",
    "unidecode>=1.3.8",
    "onnx>=1.16.2",
    "onnxruntime>=1.18.1",
    "soundfile>=0.12.0",
    "datasets",
    "huggingface_hub",
    "pydub",
]

print("\n📦 Installing model dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + SAFE_DEPS, check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"], check=False)

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

print(f"\n✅ Environment ready. Working Directory: {os.getcwd()}")

## 🗃️ Cell 4 — Robust Dataset Download, Extraction & Audio Sanitization

Imports dataset, extracts archives, converts all audio formats (`mp3`, `flac`, `ogg`, `m4a`, `wav`), and **actively filters out corrupted files, zero-byte audio, or unreadable frames**.

In [ ]:
#@title 🗃️ Download & Sanitize Audio Dataset {display-mode: "form"}

import os, glob, json, shutil, subprocess, traceback
from pathlib import Path
import soundfile as sf
import numpy as np

def touch_marker(path):
    Path(path).touch()

def marker_exists(path):
    return os.path.exists(path)

def is_valid_audio(audio_path):
    """Audits audio file for corrupt headers, zero-length, or silence."""
    try:
        if not os.path.exists(audio_path) or os.path.getsize(audio_path) < 100:
            return False
        info = sf.info(audio_path)
        if info.duration < 0.1 or info.frames == 0:
            return False
        return True
    except Exception:
        return False

def convert_audio_robust(src_path, dst_path, target_sr=SAMPLE_RATE):
    """Converts any audio file to mono WAV PCM_16 at target_sr.
    Returns True if successfully converted & verified, False if corrupted.
    """
    # Try librosa first
    try:
        import librosa
        wav, _ = librosa.load(src_path, sr=target_sr, mono=True)
        if len(wav) > 0 and np.isfinite(wav).all():
            sf.write(dst_path, wav, target_sr, subtype='PCM_16')
            if is_valid_audio(dst_path):
                return True
    except Exception:
        pass
    
    # Try pydub / ffmpeg fallback
    try:
        from pydub import AudioSegment
        audio = AudioSegment.from_file(src_path)
        audio = audio.set_frame_rate(target_sr).set_channels(1).set_sample_width(2)
        audio.export(dst_path, format="wav")
        if is_valid_audio(dst_path):
            return True
    except Exception:
        pass
        
    # Try raw ffmpeg
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", str(src_path),
             "-ar", str(target_sr), "-ac", "1", "-sample_fmt", "s16", str(dst_path)],
            check=True, capture_output=True
        )
        if is_valid_audio(dst_path):
            return True
    except Exception:
        pass
        
    return False

# ══════════════════════════════════════════════════════════
# PHASE 1: Download dataset from HuggingFace
# ══════════════════════════════════════════════════════════
if marker_exists(MARKER_DOWNLOAD_DONE):
    print("⏭️  Phase 1 (Download): Already completed — skipping.")
else:
    print("📥 Phase 1: Downloading dataset...")
    try:
        from datasets import load_dataset
        load_kwargs = {"trust_remote_code": True}
        if HF_TOKEN: load_kwargs["token"] = HF_TOKEN
        try:
            ds = load_dataset(HF_DATASET_ID, **load_kwargs)
            print(f"   Dataset loaded via HuggingFace Datasets API: {ds}")
            _ds_loaded_via_lib = True
        except Exception as e_ds:
            print(f"   Datasets API direct load failed: {e_ds}. Falling back to snapshot download...")
            from huggingface_hub import snapshot_download
            snapshot_download(
                repo_id=HF_DATASET_ID, repo_type="dataset",
                local_dir=os.path.join(DRIVE_RAW_DATASET, "_hf_snapshot"),
                token=HF_TOKEN or None,
            )
            _ds_loaded_via_lib = False
            print("   ✅ Snapshot downloaded.")
        touch_marker(MARKER_DOWNLOAD_DONE)
        print("✅ Phase 1 complete.")
    except Exception as e:
        print(f"❌ Phase 1 failed: {e}")
        raise

# ══════════════════════════════════════════════════════════
# PHASE 2: Convert & Audit Audio Files
# ══════════════════════════════════════════════════════════
if marker_exists(MARKER_CONVERT_DONE):
    print("⏭️  Phase 2 (Convert & Audit): Already completed — skipping.")
else:
    print("\n🔄 Phase 2: Converting audio to WAV & filtering corrupted files...")
    converted = 0
    corrupted_skipped = 0
    metadata_rows = []
    
    if '_ds_loaded_via_lib' in dir() and _ds_loaded_via_lib:
        for split_name in ds:
            split = ds[split_name]
            audio_col = next((c for c in split.column_names if c in ('audio', 'speech', 'wav', 'sound', 'recording')), None)
            text_col  = next((c for c in split.column_names if c in ('text', 'sentence', 'transcription', 'transcript', 'utterance')), None)
            
            if not audio_col or not text_col:
                print(f"⚠️ Missing column in {split_name}. Columns: {split.column_names}")
                continue
                
            print(f"   Processing split '{split_name}' ({len(split)} samples)...")
            for idx, sample in enumerate(split):
                file_id = f"{split_name}_{idx:06d}"
                wav_path = os.path.join(DRIVE_CONVERTED_WAV, f"{file_id}.wav")
                text = str(sample[text_col] or "").strip()
                
                if not text: continue
                
                if os.path.exists(wav_path) and is_valid_audio(wav_path):
                    metadata_rows.append((file_id, text))
                    converted += 1
                    continue
                    
                audio_data = sample[audio_col]
                success = False
                if isinstance(audio_data, dict) and 'array' in audio_data:
                    try:
                        arr = np.array(audio_data['array'], dtype=np.float32)
                        sr = audio_data['sampling_rate']
                        if sr != SAMPLE_RATE:
                            import librosa
                            arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
                        sf.write(wav_path, arr, SAMPLE_RATE, subtype='PCM_16')
                        success = is_valid_audio(wav_path)
                    except Exception: success = False
                elif isinstance(audio_data, str) and os.path.isfile(audio_data):
                    success = convert_audio_robust(audio_data, wav_path)
                elif isinstance(audio_data, bytes):
                    tmp_p = f"/tmp/_tmp_{idx}.bin"
                    with open(tmp_p, 'wb') as f: f.write(audio_data)
                    success = convert_audio_robust(tmp_p, wav_path)
                    if os.path.exists(tmp_p): os.remove(tmp_p)
                    
                if success:
                    metadata_rows.append((file_id, text))
                    converted += 1
                else:
                    corrupted_skipped += 1
                    if os.path.exists(wav_path): os.remove(wav_path)
                    
                if converted % 500 == 0 and converted > 0:
                    print(f"      ... {converted} valid audio files processed")
        del ds
    else:
        snapshot_dir = os.path.join(DRIVE_RAW_DATASET, "_hf_snapshot")
        # Extract archives
        import zipfile, tarfile
        for root, _, files in os.walk(snapshot_dir):
            for f in files:
                fp = os.path.join(root, f)
                try:
                    if f.endswith('.zip'):
                        with zipfile.ZipFile(fp, 'r') as z: z.extractall(root)
                    elif f.endswith(('.tar.gz', '.tgz', '.tar')):
                        with tarfile.open(fp) as t: t.extractall(root)
                except Exception: pass
                
        AUDIO_EXT = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma'}
        audio_files = [os.path.join(r, f) for r, _, fs in os.walk(snapshot_dir) for f in fs if Path(f).suffix.lower() in AUDIO_EXT]
        print(f"   Found {len(audio_files)} audio files on disk. Verifying & converting...")
        for af in audio_files:
            stem = Path(af).stem
            dst_wav = os.path.join(DRIVE_CONVERTED_WAV, f"{stem}.wav")
            if os.path.exists(dst_wav) and is_valid_audio(dst_wav):
                converted += 1
            else:
                if convert_audio_robust(af, dst_wav):
                    converted += 1
                else:
                    corrupted_skipped += 1
                    if os.path.exists(dst_wav): os.remove(dst_wav)
                    
    meta_path = os.path.join(DRIVE_METADATA, "_all_metadata.json")
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata_rows, f, ensure_ascii=False)
        
    print(f"\n✅ Phase 2 complete. Valid files: {converted} | 🛡️ Corrupted files skipped: {corrupted_skipped}")
    touch_marker(MARKER_CONVERT_DONE)

## 📝 Cell 5 — Generate Train & Val Filelists (Filtered & Sanitized)

Pairs valid WAV audio files with their transcriptions and creates `train.csv` / `val.csv` metadata files.

In [ ]:
#@title 📝 Generate Sanitized Metadata {display-mode: "form"}

import os, json, random
from pathlib import Path

if marker_exists(MARKER_METADATA_DONE):
    print("⏭️  Phase 3 (Metadata): Already completed — skipping.")
else:
    print("📝 Phase 3: Building sanitized train/val filelists...")
    meta_path = os.path.join(DRIVE_METADATA, "_all_metadata.json")
    all_rows = json.load(open(meta_path, encoding='utf-8')) if os.path.exists(meta_path) else []
    
    # Get verified non-empty WAV stems
    valid_wav_stems = {Path(f).stem for f in os.listdir(DRIVE_CONVERTED_WAV) if f.endswith('.wav') and is_valid_audio(os.path.join(DRIVE_CONVERTED_WAV, f))}
    print(f"   Verified WAV files available on Drive: {len(valid_wav_stems)}")
    
    valid_entries = []
    if all_rows:
        for fid, text in all_rows:
            if fid in valid_wav_stems and text and text.strip():
                valid_entries.append((fid, text.strip()))
    else:
        valid_entries = [(stem, stem) for stem in sorted(valid_wav_stems)]
        
    print(f"   Paired Audio+Text samples: {len(valid_entries)}")
    if not valid_entries:
        raise RuntimeError("❌ Zero valid audio+text pairs found. Check your dataset audio integrity.")
        
    random.seed(42)
    random.shuffle(valid_entries)
    val_count = max(1, int(len(valid_entries) * 0.05))
    val_entries = valid_entries[:val_count]
    train_entries = valid_entries[val_count:]
    
    for fname, entries in [("train.csv", train_entries), ("val.csv", val_entries)]:
        fpath = os.path.join(DRIVE_RAW_DATASET, fname)
        with open(fpath, 'w', encoding='utf-8', newline='\n') as f:
            for fid, text in entries:
                f.write(f"{fid}|{text.replace('|', ' ')}\n")
        print(f"   Wrote {fname}: {len(entries)} items")
        
    touch_marker(MARKER_METADATA_DONE)
    print("✅ Phase 3 complete.")

## 🧬 Cell 6 — Preprocess Dataset (Phonemization & Feature Extraction)

Generates pitch, energy, mel-spectrogram numpy arrays (`.npz`) and phoneme JSON files (`.json`).

In [ ]:
#@title 🧬 Preprocess Phonemes & Features {display-mode: "form"}

import os, sys, subprocess, shutil
from pathlib import Path

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

if marker_exists(MARKER_PREPROCESS_DONE):
    print("⏭️  Phase 4 (Preprocess): Already completed — skipping.")
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    if not os.path.exists(LOCAL_DATA_DIR):
        os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)
    print(f"   ✅ Linked local path {LOCAL_DATA_DIR} → {DRIVE_PREPROCESSED}")
else:
    print("🧬 Phase 4: Preprocessing audio features & phonemes...")
    
    # Remove incomplete output folder if it exists
    if os.path.isdir(DRIVE_PREPROCESSED):
        shutil.rmtree(DRIVE_PREPROCESSED)
        
    cmd = [
        sys.executable, "-m", "optispeech.tools.preprocess_dataset",
        EXPERIMENT_NAME,
        DRIVE_RAW_DATASET,
        DRIVE_PREPROCESSED,
        "--n-workers", str(PREPROCESS_WORKERS),
        "--batch-size", "8",
    ]
    print(f"   Executing: {' '.join(cmd)}")
    res = subprocess.run(cmd, cwd=LOCAL_REPO)
    
    if res.returncode != 0:
        raise RuntimeError(f"❌ Preprocessing tool failed with exit code {res.returncode}")
        
    # Symlink to repo data directory
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    if os.path.exists(LOCAL_DATA_DIR):
        if os.path.islink(LOCAL_DATA_DIR): os.unlink(LOCAL_DATA_DIR)
        else: shutil.rmtree(LOCAL_DATA_DIR)
    os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)
    
    touch_marker(MARKER_PREPROCESS_DONE)
    print("✅ Phase 4 complete. Preprocessed arrays and phonemes stored on Drive.")

## 📊 Cell 7 — Calculate Data Statistics

Computes normalization statistics (`mel_mean`, `mel_std`, `pitch_mean`, `pitch_std`, `energy_mean`, `energy_std`) and updates the experiment YAML config.

In [ ]:
#@title 📊 Compute Dataset Normalization Stats {display-mode: "form"}

import os, sys, subprocess, json, re

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

if marker_exists(MARKER_STATS_DONE):
    print("⏭️  Phase 5 (Stats): Already completed — skipping.")
else:
    print("📊 Phase 5: Calculating normalization statistics...")
    cmd = [
        sys.executable, "-m", "optispeech.tools.generate_data_statistics",
        EXPERIMENT_NAME,
        "-b", "32",
        "-w", "2",
        "-o", DRIVE_DATA_STATS,
    ]
    res = subprocess.run(cmd, cwd=LOCAL_REPO)
    if res.returncode != 0:
        raise RuntimeError("❌ Failed to calculate statistics")
        
    stats_json = os.path.join(DRIVE_DATA_STATS, "stats.json")
    if os.path.exists(stats_json):
        stats = json.load(open(stats_json))
        print("\n   Calculated stats:")
        for k, v in stats.items(): print(f"     {k}: {v}")
        
        # Inject stats into experiment config
        cfg_path = os.path.join(LOCAL_REPO, "configs", "data", f"{EXPERIMENT_NAME}.yaml")
        if os.path.exists(cfg_path):
            text = open(cfg_path).read()
            for k, v in stats.items():
                text = re.sub(rf'({k}:\s*)([\d.\-]+)', rf'\g<1>{v}', text)
            with open(cfg_path, 'w') as f: f.write(text)
            print(f"   ✅ Updated data config: {cfg_path}")
            
    touch_marker(MARKER_STATS_DONE)
    print("✅ Phase 5 complete.")

## 🚀 Cell 8 — Maximum Quality GPU Training (Auto-Resume)

Trains the OptiSpeech TTS model with:
- Auto-detected mixed precision (`bf16-mixed` or `16-mixed`)
- Persistent checkpoints saved directly to Google Drive
- Seamless automatic resume from disconnection

In [ ]:
#@title 🚀 Launch Model Training {display-mode: "form"}

import os, sys, glob, subprocess
from pathlib import Path

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

# Ensure data symlink exists
if not os.path.exists(LOCAL_DATA_DIR):
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)

# Detect existing checkpoints for auto-resume
ckpts = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINTS, "**", "*.ckpt"), recursive=True), key=os.path.getmtime)
latest_ckpt = ckpts[-1] if ckpts else None

train_cmd = [
    sys.executable, "-m", "optispeech.train",
    f"experiment={EXPERIMENT_NAME}",
    f"trainer=gpu",
    f"trainer.max_steps={MAX_STEPS}",
    f"trainer.precision={OPTIMAL_PRECISION}",
    f"trainer.log_every_n_steps=10",
    f"data.batch_size={BATCH_SIZE}",
    f"data.num_workers=2",
    f"callbacks.model_checkpoint.dirpath={DRIVE_CHECKPOINTS}",
    f"callbacks.model_checkpoint.every_n_epochs=1",
    f"callbacks.model_checkpoint.save_top_k=5",
    f"callbacks.model_checkpoint.save_last=true",
    f"paths.log_dir={DRIVE_LOGS}",
]

if latest_ckpt:
    print(f"🔄 Resuming training from checkpoint: {latest_ckpt}")
    train_cmd.append(f"ckpt_path={latest_ckpt}")
else:
    print("🆕 Starting fresh model training.")

print(f"\n⚡ GPU Precision Mode: {OPTIMAL_PRECISION}")
print(f"🏋️ Command: {' '.join(train_cmd)}\n")

res = subprocess.run(train_cmd, cwd=LOCAL_REPO)
if res.returncode == 0:
    print("\n🎉 Training session finished successfully!")
else:
    print(f"\n⚠️ Training stopped (exit code {res.returncode}). Re-run this cell to resume automatically.")

## 📦 Cell 9 — Export Model to Streaming ONNX Format & Save to Drive

Exports the trained checkpoint to streaming ONNX components (`nano_encoder.onnx`, `nano_decoder_stream.onnx`, `nano_vocoder_stream.onnx`) with optional INT8 dynamic quantization.

In [ ]:
#@title 📦 Export Model to ONNX (Save to Google Drive) {display-mode: "form"}

#@markdown ### Export Options
QUANTIZE_INT8 = True  #@param {type:"boolean"}
TEST_TEXT = "\u0645\u0631\u062d\u0628\u0627\u060c \u0647\u0630\u0627 \u0627\u062e\u062a\u0628\u0627\u0631 \u0644\u0646\u0645\u0648\u0630\u062c \u0627\u0644\u062a\u062d\u0648\u064a\u0644 \u0627\u0644\u0635\u0648\u062a\u064a."  #@param {type:"string"}

import os, sys, glob, subprocess
from pathlib import Path

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

# Find best/latest checkpoint on Drive
ckpts = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINTS, "**", "*.ckpt"), recursive=True), key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError("❌ No checkpoint found in Google Drive! Please train the model first (Cell 8).")
    
checkpoint_path = ckpts[-1]
print(f"📄 Selected Checkpoint: {checkpoint_path}")
print(f"📂 Export Output Dir:   {DRIVE_ONNX_EXPORT}")

export_cmd = [
    sys.executable, "-m", "optispeech.onnx.nano_streaming",
    "--checkpoint", checkpoint_path,
    "--output-dir", DRIVE_ONNX_EXPORT,
    "--text", TEST_TEXT,
]

if QUANTIZE_INT8:
    export_cmd.append("--quantize-encoder-decoder")
    
print(f"\n🛠️ Running ONNX Export: {' '.join(export_cmd)}")
res = subprocess.run(export_cmd, cwd=LOCAL_REPO)

if res.returncode == 0:
    print("\n✅ ONNX Export Completed Successfully!")
    print(f"📁 Files saved to Google Drive: {DRIVE_ONNX_EXPORT}")
    for item in os.listdir(DRIVE_ONNX_EXPORT):
        fp = os.path.join(DRIVE_ONNX_EXPORT, item)
        sz_mb = os.path.getsize(fp) / (1024 * 1024)
        print(f"   📄 {item} ({sz_mb:.2f} MB)")
else:
    print(f"❌ Export failed with code {res.returncode}")

## 📈 Cell 10 — TensorBoard Dashboard (Optional)

In [ ]:
#@title 📈 Launch TensorBoard {display-mode: "form"}

%load_ext tensorboard
%tensorboard --logdir {DRIVE_LOGS}

## 🔍 Cell 11 — Comprehensive Project Status & Health Check

In [ ]:
#@title 🔍 Health Check & Storage Status {display-mode: "form"}

import os, glob
from datetime import datetime

print("📋 Project Health Report")
print("=" * 50)

phases = [
    ("01 Download",   MARKER_DOWNLOAD_DONE),
    ("02 Convert",    MARKER_CONVERT_DONE),
    ("03 Metadata",   MARKER_METADATA_DONE),
    ("04 Preprocess", MARKER_PREPROCESS_DONE),
    ("05 Statistics", MARKER_STATS_DONE),
]
for name, m in phases:
    print(f"   {'✅' if os.path.exists(m) else '⬜'} {name}")

wav_c = len(glob.glob(os.path.join(DRIVE_CONVERTED_WAV, "*.wav")))
print(f"\n🎵 Verified Audio Files: {wav_c}")

ckpts = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINTS, "**", "*.ckpt"), recursive=True), key=os.path.getmtime)
print(f"💾 Drive Checkpoints:   {len(ckpts)}")
for c in ckpts[-3:]:
    mb = os.path.getsize(c) / (1024 * 1024)
    print(f"   • {os.path.basename(c)} ({mb:.1f} MB)")
    
onnx_files = glob.glob(os.path.join(DRIVE_ONNX_EXPORT, "*.onnx"))
print(f"📦 ONNX Models Exported: {len(onnx_files)}")
for f in onnx_files:
    mb = os.path.getsize(f) / (1024 * 1024)
    print(f"   • {os.path.basename(f)} ({mb:.2f} MB)")